In [1]:
import json
import gc
from pathlib import Path

import torch
import torch.nn.functional as F
from PIL import Image
from tqdm import tqdm
from transformers import AutoProcessor, LlavaForConditionalGeneration

/venv/vlm-llava/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
MODEL_ID = "llava-hf/llava-1.5-7b-hf"

IMAGE_PATH = "../data/visual_genome/images/VG_100K/3.jpg"
PROMPT = "Describe this image."

MAX_NEW_TOKENS = 128

# OPERA default-like parameters
NUM_ATTN_CANDIDATES = 5
SCALE_FACTOR = 50.0
THRESHOLD = 15.0
PENALTY_WEIGHTS = 1.0

In [3]:
processor = AutoProcessor.from_pretrained(MODEL_ID)

processor.tokenizer.padding_side = "left"

if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

model = LlavaForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True,
    attn_implementation="eager",  # needed for output_attentions=True
).eval()

print("Loaded processor and model.")

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
Loading checkpoint shards: 100%|██████████| 3/3 [00:02<00:00,  1.34it/s]

Loaded processor and model.


In [4]:
def clean_output(text: str) -> str:
    text = text.strip()

    if "ASSISTANT:" in text:
        text = text.split("ASSISTANT:", 1)[-1].strip()

    return text.strip()


def load_image(path):
    return Image.open(path).convert("RGB")

In [5]:
def compute_opera_overtrust_penalty(
    attentions,
    response_start: int,
    scale_factor: float = 50.0,
    threshold: float = 15.0,
    penalty_weights: float = 1.0,
):
    """
    OPERA-style over-trust penalty.

    attentions[-1]: last decoder layer attention, shape [B, H, Q, K].
    We inspect the last generated/probed token's attention to previous response tokens.
    """

    if attentions is None:
        raise ValueError(
            "attentions is None. Make sure the model was loaded with attn_implementation='eager'."
        )

    last_layer_attn = attentions[-1]  # [B, H, Q, K]

    # Attention of current token over all previous keys.
    attn = last_layer_attn[:, :, -1, :].mean(dim=1)  # [B, K]

    seq_len = attn.shape[-1]

    # No previous generated response tokens yet.
    if response_start >= seq_len - 1:
        return torch.zeros(attn.shape[0], device=attn.device, dtype=attn.dtype)

    # Attention to previously generated response tokens, excluding current candidate token.
    response_attn = attn[:, response_start:seq_len - 1]

    if response_attn.numel() == 0:
        return torch.zeros(attn.shape[0], device=attn.device, dtype=attn.dtype)

    max_response_attn = response_attn.max(dim=-1).values

    overtrust = torch.relu(max_response_attn * scale_factor - threshold)

    penalty = penalty_weights * overtrust / scale_factor

    return penalty

In [6]:
@torch.inference_mode()
def opera_generate_one_llava_hf(
    model,
    processor,
    image,
    question: str = "Describe this image.",
    max_new_tokens: int = 128,
    num_attn_candidates: int = 5,
    scale_factor: float = 50.0,
    threshold: float = 15.0,
    penalty_weights: float = 1.0,
):
    model.eval()

    device = next(model.parameters()).device
    dtype = next(model.parameters()).dtype

    old_padding_side = processor.tokenizer.padding_side
    processor.tokenizer.padding_side = "left"

    if processor.tokenizer.pad_token is None:
        processor.tokenizer.pad_token = processor.tokenizer.eos_token

    prompt = f"USER: <image>\n{question}\nASSISTANT:"

    inputs = processor(
        text=prompt,
        images=image,
        return_tensors="pt",
        padding=True,
    )

    processor.tokenizer.padding_side = old_padding_side

    input_ids = inputs["input_ids"].to(device)
    pixel_values = inputs["pixel_values"].to(device=device, dtype=dtype)

    generated_ids = input_ids.clone()
    prompt_text_len = input_ids.shape[1]

    eos_token_id = model.generation_config.eos_token_id

    if eos_token_id is None:
        eos_token_ids = []
    elif isinstance(eos_token_id, int):
        eos_token_ids = [eos_token_id]
    else:
        eos_token_ids = list(eos_token_id)

    # Prompt-only forward to get initial next-token logits.
    attention_mask = torch.ones_like(generated_ids, device=device)

    outputs = model(
        input_ids=generated_ids,
        attention_mask=attention_mask,
        pixel_values=pixel_values,
        use_cache=False,
        output_attentions=True,
        return_dict=True,
    )

    if outputs.attentions is None:
        raise ValueError(
            "outputs.attentions is None. Reload with attn_implementation='eager'."
        )

    # This is the actual expanded prompt length after <image> is replaced by image tokens.
    response_start = outputs.attentions[-1].shape[-1]

    next_logits = outputs.logits[:, -1, :]

    for step in range(max_new_tokens):
        log_probs = F.log_softmax(next_logits, dim=-1)

        k = min(num_attn_candidates, log_probs.shape[-1])
        top_scores, top_tokens = torch.topk(log_probs, k=k, dim=-1)

        candidate_scores = []
        candidate_tokens = []
        candidate_outputs = []

        for cand_idx in range(k):
            cand_token = top_tokens[:, cand_idx:cand_idx + 1]
            cand_log_score = top_scores[:, cand_idx]

            cand_ids = torch.cat([generated_ids, cand_token], dim=-1)
            cand_attention_mask = torch.ones_like(cand_ids, device=device)

            cand_outputs = model(
                input_ids=cand_ids,
                attention_mask=cand_attention_mask,
                pixel_values=pixel_values,
                use_cache=False,
                output_attentions=True,
                return_dict=True,
            )

            penalty = compute_opera_overtrust_penalty(
                attentions=cand_outputs.attentions,
                response_start=response_start,
                scale_factor=scale_factor,
                threshold=threshold,
                penalty_weights=penalty_weights,
            )

            adjusted_score = cand_log_score - penalty

            candidate_scores.append(adjusted_score)
            candidate_tokens.append(cand_token)
            candidate_outputs.append(cand_outputs)

        candidate_scores = torch.stack(candidate_scores, dim=1)  # [1, K]
        best_idx = int(torch.argmax(candidate_scores, dim=-1).item())

        next_token = candidate_tokens[best_idx]
        selected_outputs = candidate_outputs[best_idx]

        generated_ids = torch.cat([generated_ids, next_token], dim=-1)

        next_logits = selected_outputs.logits[:, -1, :]

        if len(eos_token_ids) > 0 and int(next_token.item()) in eos_token_ids:
            break

    new_tokens = generated_ids[:, prompt_text_len:]

    answer = processor.batch_decode(
        new_tokens,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True,
    )[0]

    return clean_output(answer)

In [7]:
import json
import gc
from pathlib import Path

import torch
from PIL import Image
from tqdm import tqdm

In [8]:
# =========================
# Visual Genome full config
# =========================

VG_ROOT = Path("../data/visual_genome")
VG_OBJECTS_PATH = Path(VG_ROOT / "VisualGenome_task" / "objects.json")
OUTPUT_PATH = Path("../results/infer/visual_genome/llava15/opera.json")
FAILED_PATH = Path("../results/infer/visual_genome/llava15/opera_failed.json")

PROMPT = "Describe this image."

NUM_IMAGES = 100
MAX_NEW_TOKENS = 128

NUM_ATTN_CANDIDATES = 5
SCALE_FACTOR = 50.0
THRESHOLD = 15.0
PENALTY_WEIGHTS = 1.0

SAVE_EVERY = 1
RESUME = True

In [9]:
def resolve_vg_image_path(vg_root: Path, image_id: int) -> Path:
    candidates = [
        vg_root / "images2" / "VG_100K_2" / f"{image_id}.jpg",
        vg_root / "images" / "VG_100K" / f"{image_id}.jpg",
        vg_root / f"{image_id}.jpg",
    ]

    for path in candidates:
        if path.exists():
            return path

    raise FileNotFoundError(
        f"Could not find image {image_id}.jpg. Tried:\n"
        + "\n".join(str(p) for p in candidates)
    )


def load_image(path: Path):
    return Image.open(path).convert("RGB")


def save_json_atomic(path, rows):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    tmp_path = path.with_suffix(path.suffix + ".tmp")

    with open(tmp_path, "w", encoding="utf-8") as f:
        json.dump(rows, f, indent=2, ensure_ascii=False)

    tmp_path.replace(path)

In [10]:
with open(VG_OBJECTS_PATH, "r", encoding="utf-8") as f:
    vg_objects = json.load(f)

vg_objects = vg_objects[:NUM_IMAGES]
image_ids = [int(obj["image_id"]) for obj in vg_objects]

print(f"Loaded {len(image_ids)} Visual Genome image IDs.")
print("First 5 IDs:", image_ids[:5])

Loaded 100 Visual Genome image IDs.
First 5 IDs: [1, 2, 3, 4, 5]


In [ ]:
model.eval()

results = []
failed = []

if RESUME and OUTPUT_PATH.exists():
    with open(OUTPUT_PATH, "r", encoding="utf-8") as f:
        results = json.load(f)

    done_ids = set(int(row["image_id"]) for row in results)
    image_ids = [image_id for image_id in image_ids if image_id not in done_ids]

    print(f"Resume mode: loaded {len(results)} existing captions.")
    print(f"Remaining images: {len(image_ids)}")

if RESUME and FAILED_PATH.exists():
    with open(FAILED_PATH, "r", encoding="utf-8") as f:
        failed = json.load(f)

    failed_ids = set(int(row["image_id"]) for row in failed)
    image_ids = [image_id for image_id in image_ids if image_id not in failed_ids]

    print(f"Loaded {len(failed)} previous failures.")
    print(f"Remaining images after excluding failures: {len(image_ids)}")


for step, image_id in enumerate(tqdm(image_ids, desc="Visual Genome OPERA 100 inference"), start=1):
    image = None

    try:
        image_path = resolve_vg_image_path(VG_ROOT, image_id)
        image = load_image(image_path)

        caption = opera_generate_one_llava_hf(
            model=model,
            processor=processor,
            image=image,
            question=PROMPT,
            max_new_tokens=MAX_NEW_TOKENS,
            num_attn_candidates=NUM_ATTN_CANDIDATES,
            scale_factor=SCALE_FACTOR,
            threshold=THRESHOLD,
            penalty_weights=PENALTY_WEIGHTS,
        )

        results.append({
            "image_id": int(image_id),
            "path": str(image_path),
            "text": caption,
        })

    except Exception as e:
        failed.append({
            "image_id": int(image_id),
            "error": repr(e),
        })

        print(f"[FAILED] image_id={image_id}: {repr(e)}")

    if step % SAVE_EVERY == 0:
        results = sorted(results, key=lambda x: int(x["image_id"]))
        failed = sorted(failed, key=lambda x: int(x["image_id"]))

        save_json_atomic(OUTPUT_PATH, results)
        save_json_atomic(FAILED_PATH, failed)

    if image is not None:
        del image

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


results = sorted(results, key=lambda x: int(x["image_id"]))
failed = sorted(failed, key=lambda x: int(x["image_id"]))

save_json_atomic(OUTPUT_PATH, results)
save_json_atomic(FAILED_PATH, failed)

print(f"Saved {len(results)} captions to {OUTPUT_PATH}")
print(f"Saved {len(failed)} failures to {FAILED_PATH}")